<a href="https://colab.research.google.com/github/Tamyrhuana/AT1-UC15/blob/main/analise_candy_power_ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise do dataset Candy Power Ranking (FiveThirtyEight)

**Desafio da semana** — análise de um arquivo CSV disponibilizado no repositório [fivethirtyeight/data](https://github.com/fivethirtyeight/data) no GitHub.

**Dataset escolhido:** `candy-power-ranking` — contém dados sobre 85 doces de Halloween, com atributos como sabor, ingredientes, preço e uma "taxa de vitória" calculada a partir de 269.000 comparações entre pares de doces (qual doce as pessoas preferem quando colocado lado a lado com outro).

Fonte original: https://github.com/fivethirtyeight/data/tree/master/candy-power-ranking

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 2. Carregamento dos dados

O arquivo CSV é lido diretamente da URL "raw" do GitHub, sem necessidade de baixar o arquivo manualmente.

In [ ]:
URL_DADOS = "https://raw.githubusercontent.com/fivethirtyeight/data/master/candy-power-ranking/candy-data.csv"


def carregar_dados(url):
    """Carrega o CSV a partir de uma URL e devolve um DataFrame do pandas."""
    df = pd.read_csv(url)
    return df


candy = carregar_dados(URL_DADOS)
candy.head()

## 3. Entendendo a estrutura dos dados

Antes de analisar, é importante entender o que cada coluna representa:

| Coluna | Descrição |
|---|---|
| competitorname | Nome do doce |
| chocolate | Contém chocolate? (1 = sim, 0 = não) |
| fruity | É sabor de fruta? |
| caramel | Contém caramelo? |
| peanutyalmondy | Contém amendoim/amêndoa? |
| nougat | Contém nougat? |
| crispedricewafer | Contém arroz crocante/wafer? |
| hard | É um doce duro (tipo bala)? |
| bar | É uma barra de chocolate? |
| pluribus | Vem em pacote com vários doces? |
| sugarpercent | Percentil de açúcar (quanto mais perto de 1, mais açúcar em relação aos demais) |
| pricepercent | Percentil de preço |
| winpercent | Percentual de vitórias em comparações diretas com outros doces |

In [ ]:
def resumo_geral(df):
    """Imprime um resumo geral do DataFrame: dimensões, tipos e estatísticas descritivas."""
    print(f"Número de doces (linhas): {df.shape[0]}")
    print(f"Número de colunas: {df.shape[1]}")
    print("\nTipos de dados:")
    print(df.dtypes)
    print("\nValores ausentes por coluna:")
    print(df.isnull().sum())


resumo_geral(candy)

In [ ]:
candy.describe()

## 4. Funções reutilizáveis de análise

Aqui estão as funções principais do notebook — construídas para serem reutilizadas em diferentes colunas e perguntas, sem precisar reescrever a lógica toda vez (conforme a recomendação da atividade).

In [ ]:
def top_n_por_coluna(df, coluna, n=10, ascendente=False):
    """Retorna os N doces com maior (ou menor) valor em uma coluna específica."""
    colunas_exibidas = ["competitorname", coluna]
    return df[colunas_exibidas].sort_values(by=coluna, ascending=ascendente).head(n)


def media_winpercent_por_atributo(df, atributo):
    """Compara a média de winpercent entre doces que TÊM e NÃO TÊM um determinado atributo binário
    (ex.: chocolate, fruity, caramel, etc.)."""
    medias = df.groupby(atributo)["winpercent"].mean()
    medias.index = ["Não tem" if i == 0 else "Tem" for i in medias.index]
    return medias


def grafico_barras(dados, titulo, xlabel, ylabel, cor="#5B8DB8"):
    """Gera um gráfico de barras genérico a partir de uma Series do pandas."""
    dados.plot(kind="bar", color=cor)
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def grafico_dispersao(df, coluna_x, coluna_y, titulo):
    """Gera um gráfico de dispersão (scatter plot) entre duas colunas numéricas."""
    plt.scatter(df[coluna_x], df[coluna_y], alpha=0.7, color="#5B8DB8")
    plt.title(titulo)
    plt.xlabel(coluna_x)
    plt.ylabel(coluna_y)
    plt.tight_layout()
    plt.show()

## 5. Quais doces têm a maior "taxa de vitória" (winpercent)?

In [ ]:
top_10_mais_populares = top_n_por_coluna(candy, "winpercent", n=10)
top_10_mais_populares

In [ ]:
grafico_barras(
    top_10_mais_populares.set_index("competitorname")["winpercent"],
    titulo="Top 10 doces mais populares",
    xlabel="Doce",
    ylabel="Taxa de vitória (%)",
)

## 6. Ter chocolate faz diferença na popularidade?

Usando a mesma função `media_winpercent_por_atributo`, dá pra comparar diferentes atributos sem reescrever a lógica de comparação.

In [ ]:
media_chocolate = media_winpercent_por_atributo(candy, "chocolate")
print(media_chocolate)

grafico_barras(
    media_chocolate,
    titulo="Taxa de vitória média: doces com e sem chocolate",
    xlabel="Contém chocolate?",
    ylabel="Taxa de vitória média (%)",
)

## 7. E o sabor de fruta, também influencia?

Reaproveitando a mesma função para outro atributo.

In [ ]:
media_fruity = media_winpercent_por_atributo(candy, "fruity")
print(media_fruity)

grafico_barras(
    media_fruity,
    titulo="Taxa de vitória média: doces de fruta vs. outros",
    xlabel="É sabor de fruta?",
    ylabel="Taxa de vitória média (%)",
    cor="#C97B84",
)

## 8. Existe relação entre preço e popularidade?

Aqui usamos a função de gráfico de dispersão para verificar visualmente se doces mais caros tendem a ser mais populares.

In [ ]:
grafico_dispersao(
    candy,
    coluna_x="pricepercent",
    coluna_y="winpercent",
    titulo="Relação entre preço e taxa de vitória",
)

correlacao = candy["pricepercent"].corr(candy["winpercent"])
print(f"Correlação entre preço e popularidade: {correlacao:.2f}")

## 9. Comparando todos os atributos de sabor/ingrediente de uma vez

Reaproveitando a função `media_winpercent_por_atributo` em um laço (loop), para não repetir código manualmente para cada atributo.

In [ ]:
atributos = [
    "chocolate", "fruity", "caramel", "peanutyalmondy",
    "nougat", "crispedricewafer", "hard", "bar", "pluribus",
]


def impacto_de_cada_atributo(df, lista_atributos):
    """Calcula, para cada atributo binário da lista, a diferença de winpercent médio
    entre doces que têm e não têm aquele atributo. Devolve um DataFrame ordenado
    pelo impacto (do que mais aumenta a popularidade para o que mais reduz)."""
    resultados = []
    for atributo in lista_atributos:
        medias = media_winpercent_por_atributo(df, atributo)
        diferenca = medias["Tem"] - medias["Não tem"]
        resultados.append({"atributo": atributo, "diferenca_winpercent": diferenca})
    return pd.DataFrame(resultados).sort_values(by="diferenca_winpercent", ascending=False)


impacto = impacto_de_cada_atributo(candy, atributos)
impacto

In [ ]:
grafico_barras(
    impacto.set_index("atributo")["diferenca_winpercent"],
    titulo="Impacto de cada atributo na popularidade do doce",
    xlabel="Atributo",
    ylabel="Diferença na taxa de vitória (p.p.)",
    cor="#7FA37F",
)

## 10. Conclusões

- O atributo com maior impacto positivo na popularidade dos doces é o chocolate — doces que contêm chocolate têm, em média, uma taxa de vitória bem mais alta.
- Doces de fruta (`fruity`) tendem a ter desempenho pior nas comparações diretas.
- A correlação entre preço (`pricepercent`) e popularidade (`winpercent`) é fraca, o que sugere que o preço, isoladamente, não é um bom previsor de quão popular um doce será — sabor e ingredientes parecem pesar mais.
- O uso de funções reutilizáveis (`top_n_por_coluna`, `media_winpercent_por_atributo`, `grafico_barras`, `grafico_dispersao`, `impacto_de_cada_atributo`) permitiu repetir a mesma análise para diferentes colunas do dataset sem duplicar código.